## Importando librerías

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.optimizers.legacy import Adam
from tensorflow.keras.losses import BinaryCrossentropy

## Carga y preprocesado de datos

In [2]:
data = pd.read_csv("Datos/Titanic-Dataset.csv")

In [3]:
data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [4]:
def preprocess_dataset(df, categorical_cols, numerical_cols):
    
    titles_dict = {'Capt': 'Other',
               'Major': 'Other',
               'Jonkheer': 'Other',
               'Don': 'Other',
               'Sir': 'Other',
               'Dr': 'Other',
               'Rev': 'Other',
               'Countess': 'Other',
               'Dona': 'Other',
               'Mme': 'Mrs',
               'Mlle': 'Miss',
               'Ms': 'Miss',
               'Mr': 'Mr',
               'Mrs': 'Mrs',
               'Miss': 'Miss',
               'Master': 'Master',
               'Lady': 'Other'}
    
    df['Title'] = df['Name'].str.extract('([A-Za-z]+)\.', expand=False)
    df['Title'] = df['Title'].map(titles_dict) # Creación de la columna 'Title' a partir de los títulos extraídos del nombre
    df['Title'].fillna('Mr', inplace=True)
    means = df.groupby('Title')['Age'].mean().to_dict()
    df.loc[df.Age.isna(),'Age'] = df['Title'].loc[df.Age.isna()].map(means)
    df.Embarked = df.Embarked.fillna('S')
    df["HasCabin"] = df["Cabin"].notna().astype(int) # Creación de la columna 'HasCabin' indicando si el pasajero tenía una cabina asignada
    df['FamilySize'] = df['SibSp'] + df['Parch'] # Creación de la columna 'FamilySize' sumando el número de hermanos/cónyuges y padres/hijos a bordo
    df.drop('SibSp', axis=1, inplace=True)
    df.drop('Parch', axis=1, inplace=True)
    
    # Selección de columnas relevantes para el modelo
    df = df[["Survived",
             "Pclass",
             "Sex",
             "Age",
             "Fare",
             "Embarked",
             "Title",
             "HasCabin",
             "FamilySize"]]
    
    ohe = OneHotEncoder(sparse_output=False)
    scaler = StandardScaler()

    X_num = scaler.fit_transform(df[numerical_cols])
    X_cat = ohe.fit_transform(df[categorical_cols])

    X = np.concatenate([X_num, X_cat], axis=1)
    X = X.astype("float32")

    return ohe, scaler, X


In [5]:
# Categorías numericas y categóricas
numerical_cols = ["Age", 
                  "Fare", 
                  "FamilySize"]

categorical_cols = ["Survived", 
                    "Pclass",
                    "Sex",
                    "Embarked",
                    "Title",
                    "HasCabin"]

In [6]:
ohe, scaler, X = preprocess_dataset(data, categorical_cols=categorical_cols, numerical_cols=numerical_cols)

/var/folders/hc/mb3n8ydj71l7sft8wld706100000gn/T/ipykernel_7235/1122125085.py:23: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Title'].fillna('Mr', inplace=True)


## Parámetros para la GAN

In [7]:
NOISE_DIM = 128
CAT_DIMS = [len(ohe.categories_[i]) for i in range(len(categorical_cols))]
CAT_COLS = sum(CAT_DIMS)
NUM_COLS = len(numerical_cols)
INPUT_DIM = X.shape[1]

In [8]:
# Generador
def build_generator(noise_dim, num_cols, cat_dims):
    input_noise = layers.Input(shape=(noise_dim,)) # Input de ruido aleatorio
    
    x = layers.Dense(64, activation="leaky_relu")(input_noise) # Capa oculta con activación ReLU
    x = layers.BatchNormalization()(x)
    x = layers.Dense(32, activation="leaky_relu")(x) # Segunda capa oculta
    x = layers.BatchNormalization()(x)
    x = layers.Dense(16, activation="leaky_relu")(x) # Tercera capa oculta
    x = layers.BatchNormalization()(x)

    out_num = layers.Dense(num_cols, activation="linear", name="numeric_out")(x) # Salida numérica con activación lineal
    out_cats = []
    for i, dim in enumerate(cat_dims):
        out_cat = layers.Dense(dim, activation="softmax", name=f"cat_out_{i}")(x) # Salida categórica con activación softmax
        out_cats.append(out_cat)
    combined_output = layers.Concatenate()([out_num, *out_cats]) # Combinar salidas numéricas y categóricas
    
    model = Model(inputs=input_noise, outputs=combined_output, name="Generator")
    return model

generator = build_generator(NOISE_DIM, NUM_COLS, CAT_DIMS)

2026-02-18 18:33:10.823270: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4
2026-02-18 18:33:10.823292: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2026-02-18 18:33:10.823297: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.92 GB
2026-02-18 18:33:10.823317: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:306] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-02-18 18:33:10.823327: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:272] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [9]:
# Discriminador
def build_discriminator(input_dim):
    input_data = layers.Input(shape=(input_dim,)) # Input de datos (reales o generados)
    
    x = layers.Dense(64, activation="leaky_relu")(input_data) # Capa oculta con activación Leaky ReLU
    x = layers.Dropout(0.2)(x)
    x = layers.Dense(32, activation="leaky_relu")(x) # Segunda capa oculta
    x = layers.Dropout(0.2)(x)
    
    output = layers.Dense(1, activation="sigmoid")(x) # Salida de probabilidad con activación sigmoide
    
    model = Model(inputs=input_data, outputs=output, name="Discriminator")
    return model

discriminator = build_discriminator(INPUT_DIM)

In [10]:
# Arquitectura del GAN
class TabularGAN(Model):
    def __init__(self, discriminator, generator, noise_dim):
        super(TabularGAN, self).__init__()
        self.discriminator = discriminator 
        self.generator = generator
        self.noise_dim = noise_dim

    def compile(self, d_optimizer, g_optimizer, loss_fn):
        super(TabularGAN, self).compile()
        self.d_optimizer = d_optimizer
        self.g_optimizer = g_optimizer
        self.loss_fn = loss_fn

    def train_step(self, real_data):
        batch_size = tf.shape(real_data)[0]
        
        # Entrenar el Discriminador
        noise = tf.random.normal(shape=(batch_size, self.noise_dim)) # Generar ruido aleatorio
        fake_data = self.generator(noise) # Generar datos falsos a partir del ruido
        
        # Etiquetas: 1 para real, 0 para falso
        labels_real = tf.ones((batch_size, 1)) * 0.95 # Etiqueta suavizada para reales
        labels_fake = tf.ones((batch_size, 1)) * 0.05 # Etiqueta suavizada para falsos
        labels = tf.concat([labels_real, labels_fake], axis=0) # Combinar etiquetas reales y falsas
        
        with tf.GradientTape() as tape:
            pred_real = self.discriminator(real_data) # Predicciones del discriminador para datos reales
            pred_fake = self.discriminator(fake_data) # Predicciones del discriminador para datos falsos
            pred = tf.concat([pred_real, pred_fake], axis=0) # Combinar predicciones reales y falsas
            
            d_loss = self.loss_fn(labels, pred) # Pérdida del discriminador basada en las etiquetas y predicciones
            
        grads = tape.gradient(d_loss, self.discriminator.trainable_weights)
        self.d_optimizer.apply_gradients(zip(grads, self.discriminator.trainable_weights))

        # Entrenar el Generador
        # Queremos engañar al discriminador, así que etiquetamos los fakes como 1 (real)
        misleading_labels = tf.ones((batch_size, 1))

        with tf.GradientTape() as tape:
            fake_data = self.generator(noise) # Generar datos falsos a partir del mismo ruido
            pred_fake = self.discriminator(fake_data) # Predicciones del discriminador para los datos falsos generados
            g_loss = self.loss_fn(misleading_labels, pred_fake) # Pérdida del generador basada en la capacidad de engañar al discriminador
            
        grads = tape.gradient(g_loss, self.generator.trainable_weights)
        self.g_optimizer.apply_gradients(zip(grads, self.generator.trainable_weights))

        return {"d_loss": d_loss, "g_loss": g_loss}

## Entrenamiento

In [11]:
# Instanciar y compilar
gan = TabularGAN(discriminator=discriminator, generator=generator, noise_dim=NOISE_DIM) # Instancia del GAN con el discriminador, generador y dimensión de ruido
gan.compile(
    d_optimizer=Adam(learning_rate=5e-3),
    g_optimizer=Adam(learning_rate=5e-3),
    loss_fn=BinaryCrossentropy()
)

# Entrenar
gan.fit(X, epochs=200, batch_size=256, verbose=1) 
print("Entrenamiento finalizado.")

Epoch 1/200


2026-02-18 18:33:11.452872: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


4/4 [==============================] - 1s 117ms/step - d_loss: 0.7012 - g_loss: 0.7245
Epoch 2/200
4/4 [==============================] - 0s 12ms/step - d_loss: 0.6462 - g_loss: 0.9146
Epoch 3/200
4/4 [==============================] - 0s 13ms/step - d_loss: 0.5072 - g_loss: 1.4678
Epoch 4/200
4/4 [==============================] - 0s 12ms/step - d_loss: 0.5804 - g_loss: 1.1658
Epoch 5/200
4/4 [==============================] - 0s 11ms/step - d_loss: 0.7242 - g_loss: 0.9414
Epoch 6/200
4/4 [==============================] - 0s 11ms/step - d_loss: 1.0312 - g_loss: 0.6475
Epoch 7/200
4/4 [==============================] - 0s 10ms/step - d_loss: 1.2360 - g_loss: 0.6415
Epoch 8/200
4/4 [==============================] - 0s 10ms/step - d_loss: 0.6925 - g_loss: 2.5728
Epoch 9/200
4/4 [==============================] - 0s 10ms/step - d_loss: 0.3853 - g_loss: 4.0734
Epoch 10/200
4/4 [==============================] - 0s 10ms/step - d_loss: 0.3596 - g_loss: 2.5826
Epoch 11/200
4/4 [============

## Generación de datos sintéticos

In [12]:

def generate_synthetic_data(num):
    # Generar datos sintéticos brutos
    noise = tf.random.normal(shape=(num, NOISE_DIM))
    generated_raw = generator.predict(noise)

    # Separar numéricos y categóricos
    gen_num_scaled = generated_raw[:, :NUM_COLS]
    gen_cat_probs = generated_raw[:, NUM_COLS:]

    # Revertir escalado numérico
    gen_num = scaler.inverse_transform(gen_num_scaled)

    gen_cat = [[] for _ in range(num)]

    # Convertir probabilidades categóricas a índices y luego a categorías originales
    for i, dim in enumerate(CAT_DIMS):
        start_idx = sum(CAT_DIMS[:i])
        end_idx = start_idx + dim
        cat_probs = gen_cat_probs[:, start_idx:end_idx]
        cat_indices = np.argmax(cat_probs, axis=1)
        for i, idx in enumerate(cat_indices):
            aux = [0 for _ in range(dim)]
            aux[idx] = 1
            gen_cat[i] += aux
    
    gen_cat = ohe.inverse_transform(gen_cat)

    # Concatenar numéricos y categóricos para obtener el dataset sintético completo
    generated_data = np.concatenate([gen_num, gen_cat], axis=1)
    
    return generated_data

pd.DataFrame(generate_synthetic_data(10), columns = numerical_cols + categorical_cols).head(10)

1/1 [==============================] - 0s 102ms/step


,Age,Fare,FamilySize,Survived,Pclass,Sex,Embarked,Title,HasCabin
0,11.715076,53.226341,0.933232,0,3,male,S,Mr,0
1,32.859222,34.851578,0.618968,0,3,male,S,Mr,0
2,12.598082,54.511597,0.965544,0,3,male,S,Mr,0
3,33.86256,38.070343,0.861374,0,3,male,S,Mr,0
4,33.544601,87.001999,1.059978,0,1,male,S,Mr,1
5,31.438448,36.264442,0.673428,0,3,male,S,Mr,0
6,29.377378,44.085743,0.847661,0,3,male,S,Mr,0
7,19.334831,55.170666,1.200937,0,3,male,S,Mr,0
8,5.874833,61.81665,1.43644,0,3,male,S,Mr,0
9,33.426453,49.221439,0.678579,0,3,male,S,Mr,0


Los datos generados parecen bastante decentes en su mayoría, sin embargo, se están generando datos ilógicos para ciertas escalas númericas, además parece haber una tendencia a la repetición de datos categóricos. Nuestro generador podría ser mejor tal vez con un enfoque distinto en la arquitectura o en las funciones de activación.